# Llibreries

In [53]:
import pandas as pd
import numpy as np
from zipfile import ZipFile
import os
import sys
from pathlib import Path

# Visualització
import matplotlib.pyplot as plt
import seaborn as sns

# Config vis
sns.set_theme()

# Funcions
cwd = os.getcwd()
parent = os.path.abspath(os.path.join(cwd, os.pardir))
sys.path.insert(0, parent)
from src.utils import neteja_noms_columnes

# Dimensions
Carrega del dataset que conté totes les dimensions de les dades

In [2]:
dimensions = pd.read_csv("../data/dimensions/pad_dimensions.csv")
dim_barris = pd.read_csv("../data/dimensions/BarcelonaCiutat_Barris.csv")

## Dades Urbanes
En aquest apartat s'ingestaran i transformaran les dades de caracter urbà. 

### Incidents Urbans

En alguns articles, s' introdueixen variables de criminalitat per estudiar la gentrificació. Si bé no s' ha pogut aconseguir índexs de criminalitats ja creats, es pot agregar informació sobre incidents a nivell de barri. 

#### 2025

In [4]:
df_incidents_25_raw = pd.read_csv("../data/raw/urbanes/2025_incidents_gestionats_gub.csv")
df_incidents_25_raw.head()

,Codi_Incident,Descripcio_Incident,Codi_districte,Nom_districte,Codi_barri,Nom_barri,NK_Any,Mes_any,Nom_mes,Numero_incidents_GUB
0,000,INCENDIS,NaN,NaN,NaN,NaN,2025,1,gener,2
1,000,INCENDIS,NaN,NaN,NaN,NaN,2025,2,febrer,8
2,000,INCENDIS,NaN,NaN,NaN,NaN,2025,3,març,1
3,000,INCENDIS,NaN,NaN,NaN,NaN,2025,4,abril,4
4,000,INCENDIS,NaN,NaN,NaN,NaN,2025,5,maig,3


In [5]:
df_incidents_25_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29999 entries, 0 to 29998
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Codi_Incident         29999 non-null  object 
 1   Descripcio_Incident   29999 non-null  object 
 2   Codi_districte        29640 non-null  float64
 3   Nom_districte         29640 non-null  object 
 4   Codi_barri            29640 non-null  float64
 5   Nom_barri             29640 non-null  object 
 6   NK_Any                29999 non-null  int64  
 7   Mes_any               29999 non-null  int64  
 8   Nom_mes               29999 non-null  object 
 9   Numero_incidents_GUB  29999 non-null  int64  
dtypes: float64(2), int64(3), object(5)
memory usage: 2.3+ MB


**Observacions:**
- Tal i com es pot observar, en aquest dataset existeixen valors nuls. 
- Els formats de les dades semblen adequats.


Els nuls semblen estar acumulats en els atributs amb informació sobre districte i barri. Donada la importància de no tenir aquesta dada buida, començarem eliminant les files on almenys codi_barri és NaN.

In [16]:
df_incidents_25_mod  = df_incidents_25_raw.dropna(subset=["Codi_barri"])
df_incidents_25_mod.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29640 entries, 11 to 29998
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Codi_Incident         29640 non-null  object 
 1   Descripcio_Incident   29640 non-null  object 
 2   Codi_districte        29640 non-null  float64
 3   Nom_districte         29640 non-null  object 
 4   Codi_barri            29640 non-null  float64
 5   Nom_barri             29640 non-null  object 
 6   NK_Any                29640 non-null  int64  
 7   Mes_any               29640 non-null  int64  
 8   Nom_mes               29640 non-null  object 
 9   Numero_incidents_GUB  29640 non-null  int64  
dtypes: float64(2), int64(3), object(5)
memory usage: 2.5+ MB


**Observacions:**
- Es pot observar com el nombre de registres ha disminuit, i ara no s' hi detecta cap valor nul en les dades. 
- El codi barri és un valor categòric i per tant hauria d'estar com int32 almenys.

In [22]:
df_incidents_25_mod.loc[:, "Codi_barri"] = df_incidents_25_mod["Codi_barri"].astype("int32")

In [51]:
# Agrupem les dades per tipologia d'incident i barri
df_incidents_25_grouped = df_incidents_25_mod.groupby(["Codi_barri"]).size().reset_index(name = 'total_incidents')
df_incidents_25_grouped.head()

,Codi_barri,total_incidents
0,1,561
1,2,508
2,3,491
3,4,498
4,5,479


In [123]:
df_incidents_25_grouped_clean = neteja_noms_columnes(df_incidents_25_grouped)

#### 2015

In [37]:
df_incidents_15_raw = pd.read_csv("../data/raw/urbanes/2015_incidents_gestionats_gub.csv")
df_incidents_15_raw.head()

,Codi Incident,Descripció Incident,Codi districte,Nom districte,Codi barri,Nom barri,NK Any,Mes de any,Nom mes,Número d'incidents GUB
0,222,INFRACCIONS EN MOVIMENT ...,6,Gràcia,32,el Camp d'en Grassot i Gràcia Nova,2015,5,Maig,10
1,400,CONVIVÈNCIA VEINAL ...,10,Sant Martí,69,Diagonal Mar i el Front Marítim del Poblenou,2015,5,Maig,13
2,420,INCIDÈNCIES DE LOCALS ...,2,Eixample,7,la Dreta de l'Eixample,2015,5,Maig,43
3,660,VIOLÈNCIA DOMÈSTICA ...,9,Sant Andreu,61,la Sagrera,2015,4,Abril,1
4,203,SENYALS AUTOMATITZADES ...,2,Eixample,5,el Fort Pienc,2015,4,Abril,19


In [38]:
df_incidents_15_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33801 entries, 0 to 33800
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Codi Incident           33801 non-null  object
 1   Descripció Incident     33801 non-null  object
 2   Codi districte          33801 non-null  int64 
 3   Nom districte           33801 non-null  object
 4   Codi barri              33801 non-null  int64 
 5   Nom barri               33801 non-null  object
 6   NK Any                  33801 non-null  int64 
 7   Mes de any              33801 non-null  int64 
 8   Nom mes                 33801 non-null  object
 9   Número d'incidents GUB  33801 non-null  int64 
dtypes: int64(5), object(5)
memory usage: 2.6+ MB


**Observacions:**
- No es detecten valors nuls per al conjunt d' incidents del 2015.
- Codi Barri correctament importada com int.

In [50]:
df_incidents_15_grouped = df_incidents_15_raw.groupby(["Codi barri"]).size().reset_index(name = 'total_incidents')
df_incidents_15_grouped.head()

,Codi barri,total_incidents
0,-1,564
1,1,674
2,2,625
3,3,615
4,4,592


In [48]:
df_incidents_15_grouped["Codi barri"].unique()

array([-1,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73], dtype=int64)

**Observacions:**
- S'observa un codi de barri marcat com -1. No existeix a la nostra dimensió de barris,i per tant pot referir-se al localitzacions desconegudes.

In [122]:
df_incidents_15_grouped_clean = neteja_noms_columnes(df_incidents_15_grouped)

### Comerços
En aquest apartat, integrem dades dels comerços als barris.

In [70]:
with ZipFile("../data/raw/urbanes/Taula estadística - Nombre de locals comercials actius per sector d’activitat i grup d’activitat.zip", 'r') as zip_ref:
    zip_ref.extractall("../data/raw/urbanes/locals_comercials")

In [88]:
df_locals_raw = pd.read_csv("../data/raw/urbanes/locals_comercials/Taula estadística.csv")
df_locals_raw.head()

,Territori,Tipus de territori,Temps,Activitats immobiliàries,Ensenyament,Equipaments culturals i recreatius,Finances i assegurances,"Manteniment, neteja i producció",Reparacions (Electrodomèstics i automòbils),"Restaurants, bars i hotels (Inclòs hostals, pensions i fondes)",Sanitat i assistència
0,el Raval,Barri,31 Des. 2014,12,68,41,53,26,7,544,7
1,el Raval,Barri,31 Des. 2016,12,54,66,40,-,11,469,8
2,el Raval,Barri,31 Des. 2019,10,52,62,36,-,9,540,14
3,el Raval,Barri,31 Des. 2022,8,44,59,26,-,10,500,8
4,el Raval,Barri,31 Des. 2024,6,49,55,27,-,10,537,14


**Observacions.**
- El format és diferent ja que en aquest cas les dades provenen d' una taula estadística i no de Microdades. 
- Es pot observar també que no existeixen dades del 2015 ni 2023. Utilitzarem el conjunt de 31/12/2014 com a aproximació per a les dades de 2015 i les dades de 2022 per a l'aproximació de les dades de 2023.

In [89]:
df_locals_mod = df_locals_raw.copy()

# Extreiem any
df_locals_mod["Any"] = df_locals_mod["Temps"].str.extract(r'(\d{4})')
df_locals_mod = df_locals_mod.drop(columns=["Temps", "Tipus de territori"])

In [90]:
# Convertim a format llarg
df_locals_melt = df_locals_mod.melt(id_vars=["Territori",   "Any"], var_name="tipus_local", value_name="total_locals")
df_locals_melt.head()

,Territori,Any,tipus_local,total_locals
0,el Raval,2014,Activitats immobiliàries,12
1,el Raval,2016,Activitats immobiliàries,12
2,el Raval,2019,Activitats immobiliàries,10
3,el Raval,2022,Activitats immobiliàries,8
4,el Raval,2024,Activitats immobiliàries,6


In [99]:
df_locals_melt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2920 entries, 0 to 2919
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Territori     2920 non-null   object
 1   Any           2920 non-null   object
 2   tipus_local   2920 non-null   object
 3   total_locals  2920 non-null   object
dtypes: object(4)
memory usage: 91.4+ KB


**Observacions:**
- No es detecten nuls perquè no estan com a tal. Apareixen en forma de "-". Per tant s' han d' eliminar aquests registres per poder convertir el valor a int.

In [91]:
print("Tipos de locals: \n", df_locals_melt["tipus_local"].unique())

Tipos de locals: 
 ['Activitats immobiliàries' 'Ensenyament'
 'Equipaments culturals i recreatius' 'Finances i assegurances'
 'Manteniment, neteja i producció'
 'Reparacions (Electrodomèstics i automòbils)'
 'Restaurants, bars i hotels (Inclòs hostals, pensions i fondes)'
 'Sanitat i assistència']


In [100]:
# Reduim el scope als anys i tipologia de locals que ens interessen
locals_scope = [
    "Restaurants, bars i hotels (Inclòs hostals, pensions i fondes)",
    "Manteniment, neteja i producció",
    "Reparacions (Electrodomèstics i automòbils)",
    "Sanitat i assistència"
]
df_locals_filtered = df_locals_melt[
    (df_locals_melt["Any"].isin(["2014", "2022"])) & 
    (df_locals_melt["tipus_local"].isin(locals_scope)) &
    (df_locals_melt["total_locals"] != "-")
    ]

df_locals_filtered.head()

,Territori,Any,tipus_local,total_locals
1460,el Raval,2014,"Manteniment, neteja i producció",26
1465,el Barri Gòtic,2014,"Manteniment, neteja i producció",3
1470,la Barceloneta,2014,"Manteniment, neteja i producció",4
1475,"Sant Pere, Santa Caterina i la Ribera",2014,"Manteniment, neteja i producció",19
1480,el Fort Pienc,2014,"Manteniment, neteja i producció",16


Amb aquests 3 grups obtindrem dades de: 
- Locals de restauració
- Locals de serveis professionals o oficis
- Locals sanitaris

In [105]:
# Convertim a format ample per facilitar la comparativa entre anys
df_locals_filtered.loc[:, ["total_locals"]] = df_locals_filtered["total_locals"].astype("int32")

df_locals_pivot = df_locals_filtered.pivot_table(index=["Territori", "Any"], columns="tipus_local", values="total_locals", aggfunc="sum", fill_value=0).reset_index()
df_locals_pivot.head()

tipus_local,Territori,Any,"Manteniment, neteja i producció",Reparacions (Electrodomèstics i automòbils),"Restaurants, bars i hotels (Inclòs hostals, pensions i fondes)",Sanitat i assistència
0,Baró de Viver,2014,0,0,6,0
1,Baró de Viver,2022,0,0,9,2
2,Can Baró,2014,1,12,17,1
3,Can Baró,2022,0,8,19,1
4,Can Peguera,2014,0,0,2,1


In [114]:
# Obtenim el codi del barri a partir del nom
df_locals_code = df_locals_pivot.merge(dim_barris[["codi_barri", "nom_barri"]], left_on="Territori", right_on="nom_barri", how="left")\
    .drop(columns=["nom_barri", "Territori"])

# Mostrem si hi ha valors sense codi de barri
df_locals_code[df_locals_code["codi_barri"].isna()]

,Any,"Manteniment, neteja i producció",Reparacions (Electrodomèstics i automòbils),"Restaurants, bars i hotels (Inclòs hostals, pensions i fondes)",Sanitat i assistència,codi_barri


In [118]:
# Afegim les columnes amb les tipologies agrupades
df_locals_code["locals_serveis_professionals"] = df_locals_code["Manteniment, neteja i producció"] + df_locals_code["Reparacions (Electrodomèstics i automòbils)"]
df_locals_final = df_locals_code\
    .rename(columns={"Restaurants, bars i hotels (Inclòs hostals, pensions i fondes)": "locals_restauracio", "Sanitat i assistència": "locals_sanitaris"})\
    .drop(columns=["Manteniment, neteja i producció", "Reparacions (Electrodomèstics i automòbils)"])

In [120]:
# Separem els conjunts de dades per any
df_locals_22 = df_locals_final[df_locals_final["Any"] == "2022"].drop(columns=["Any"])
df_locals_14 = df_locals_final[df_locals_final["Any"] == "2014"].drop(columns=["Any"])

# Agregacions
Un cop hem tractat les diferents dades urbanes, procedim a agrupar-les al seu any corresponent.

In [126]:
df_urbanes = dim_barris[["codi_barri"]].copy()

df_urbanes_25 = df_urbanes\
                        .merge(df_incidents_25_grouped_clean, on="codi_barri", how="left")\
                        .merge(df_locals_22, on="codi_barri", how="left")

df_urbanes_25.head()

,codi_barri,total_incidents,locals_restauracio,locals_sanitaris,locals_serveis_professionals
0,1,561,500,8,10
1,2,508,440,6,2
2,3,491,177,15,5
3,4,498,387,13,2
4,5,479,200,23,33


In [127]:
df_urbanes_15 = df_urbanes\
                        .merge(df_incidents_15_grouped_clean, on="codi_barri", how="left")\
                        .merge(df_locals_14, on="codi_barri", how="left")

df_urbanes_15.head()

,codi_barri,total_incidents,locals_restauracio,locals_sanitaris,locals_serveis_professionals
0,1,674,544,7,33
1,2,625,517,4,5
2,3,615,218,8,6
3,4,592,374,8,22
4,5,564,174,16,55


In [128]:
# Guardem els dataframes processats
df_urbanes_25.to_csv("../data/processed/df_urbanes_25.csv", index=False)
df_urbanes_15.to_csv("../data/processed/df_urbanes_15.csv", index=False)